# ⚡ LLM Forge Studio - Colab Edition
### No-Code LLM Factory - From Scratch + Fine-Tuning - One-Click in Colab!

<p align="center">
  <a href="https://github.com/arrehmanrehman28-byte/llm-forge-studio"><img src="https://img.shields.io/badge/LLM-Forge%20Studio-purple?style=for-the-badge" /></a>
  <a href="https://colab.research.google.com/github/arrehmanrehman28-byte/llm-forge-studio/blob/main/colab.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" /></a>
</p>

**✅ FIXED! Clone URL now correct!** Build custom LLMs in 5 minutes - No code! QLoRA 7B on 8GB VRAM, From Scratch, GGUF/Ollama export.

---
## 🚀 One-Click Setup (Run all cells in order!)


### Step 1: Check GPU (Free T4 in Colab!)

In [ ]:
!nvidia-smi
import torch
print(f"Torch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print("✅ Ready for Qwen 0.5B, SmolLM2 360M, tiny-10M!")
else:
    print("⚠️ CPU mode - Use tiny-10M (2h) or Qwen 0.5B (5 mins)")


### Step 2: Clone Repo & Install (1 minute) - FIXED URL!

In [ ]:
# ✅ FIXED: No dependency conflicts + Python 3.13 compatible
!rm -rf llm-forge-studio
!git clone https://github.com/arrehmanrehman28-byte/llm-forge-studio.git
%cd llm-forge-studio
print("✅ Cloned!")

print("📦 Installing dependencies (fixed dependency conflicts)...")
!pip install -q --upgrade pip
# Fix 1: Install torch (has wheels for Python 3.13)
!pip install -q torch --index-url https://download.pytorch.org/whl/cu121
# Fix 2: Install gradio 4.44.1 first (installs compatible fastapi/starlette/jinja2)
!pip install -q gradio==4.44.1
# Fix 3: Downgrade ONLY jinja2 to 3.1.2 (fixes unhashable dict) - keep fastapi/starlette compatible
!pip install -q jinja2==3.1.2 markupsafe==2.1.5
print("✅ Fixed Jinja2 3.1.2")
# Fix 4: Install ML packages WITHOUT strict pins (use latest wheels for Python 3.13)
# Avoid safetensors build error by using binary only and latest version
!pip install -q transformers peft trl accelerate datasets --no-deps
!pip install -q safetensors --only-binary=:all:
!pip install -q tokenizers huggingface_hub --upgrade
# Fix 5: CRITICAL - Downgrade huggingface_hub to 0.25.2 (fixes HfFolder) + keep jinja2 3.1.2
!pip install -q huggingface_hub==0.25.2 jinja2==3.1.2 --force-reinstall --no-deps
!pip install -q huggingface_hub==0.25.2 --force-reinstall
print("✅ Installed! Dependency conflicts fixed")

# Verify
import jinja2, huggingface_hub, gradio, safetensors
print(f"jinja2: {jinja2.__version__} (should be 3.1.2)")
print(f"huggingface_hub: {huggingface_hub.__version__} (should be 0.25.2)")
print(f"gradio: {gradio.__version__}")
print(f"safetensors: {safetensors.__version__}")
try:
    from huggingface_hub import HfFolder
    print("✅ HfFolder works!")
except Exception as e:
    print(f"❌ HfFolder: {e}")
try:
    import gradio as gr
    print("✅ Gradio works!")
except Exception as e:
    print(f"❌ Gradio: {e}")


### Step 3: Launch LLM Forge Studio (Gradio + FastAPI) - Public Link!

In [ ]:
# Fix for Colab - Patch gradio_client bool bug + localhost check + HfFolder fix
print("🔧 Patching Gradio for Colab...")

# Patch 1: Fix bool bug in gradio_client
import gradio_client.utils as client_utils
orig_func = client_utils._json_schema_to_python_type
orig_get = client_utils.get_type
def patched_get(schema):
    if isinstance(schema, bool): return "Any"
    try: return orig_get(schema)
    except: return "Any"
def patched_json(schema, defs=None):
    if isinstance(schema, bool): return "Any"
    if isinstance(schema, dict) and "additionalProperties" in schema and isinstance(schema["additionalProperties"], bool):
        schema = dict(schema)
        schema["additionalProperties"] = {"type": "object"} if schema["additionalProperties"] else {}
        if not schema["additionalProperties"]:
            del schema["additionalProperties"]
    try: return orig_func(schema, defs)
    except: return "Any"
client_utils.get_type = patched_get
client_utils._json_schema_to_python_type = patched_json
print("✅ Patched gradio_client bool bug")

# Patch 2: Fix localhost check (must be before importing gradio)
import gradio.networking
gradio.networking.is_local_port_accessible = lambda *args, **kwargs: True
print("✅ Patched networking")

# Patch 3: Fix HfFolder import error - ensure huggingface_hub 0.25.2
import huggingface_hub
print(f"huggingface_hub version: {huggingface_hub.__version__}")
if huggingface_hub.__version__ != "0.25.2":
    print("⚠️  Version mismatch, reinstalling 0.25.2...")
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub==0.25.2", "--force-reinstall"])
    print("✅ Reinstalled 0.25.2, please restart runtime and run again")

# Launch FastAPI in background
import threading, uvicorn, time
from backend.server import app as fastapi_app
def run_api():
    uvicorn.run(fastapi_app, host="0.0.0.0", port=8000, log_level="warning")
threading.Thread(target=run_api, daemon=True).start()
time.sleep(2)
print("✅ FastAPI on port 8000")

# Launch Gradio with share=True for Colab public link
from app import app as gradio_app
print("🚀 Launching LLM Forge Studio...")
print("👉 Click the Gradio public link below (https://...gradio.live)")
print("📊 Features: Welcome, Maker, From Scratch, Data, Train, Results, Test, Export")
gradio_app.launch(share=True, show_error=True, server_port=7860, server_name="0.0.0.0")


---
## 🎯 Quick Start in Colab (After Launch)

1. **Click Gradio public link** (https://xxxxx.gradio.live) - Appears after launch!
2. **Welcome Tab**: 3-step guide (30s)
3. **Maker Tab**: Choose Qwen 0.5B (fastest for T4) or tiny-10M from scratch
4. **Data Tab**: Click **Use Sample Data** button (6 examples) or upload CSV
5. **Train Tab**: Enable DoRA, NEFTune, Packing (all checked by default) → Click **TRAIN MY AI NOW!**
6. **Results Tab**: See 79.5% loss reduction, 95.5% PPL reduction
7. **Test Tab**: Chat with your model
8. **Export Tab**: GGUF Q4_K_M → Download → Ollama

---
## 🐍 Direct Code Training (No UI)


In [ ]:
# Direct training in Colab - No UI needed
from backend.trainer import train_model
from backend.efficiency import calculate_efficiency_gains

config = {
    'model_id': 'Qwen/Qwen2-0.5B-Instruct',
    'epochs': 2,
    'batch_size': 4,
    'lr': 2e-4,
    'lora_r': 16,
    'use_lora': True,
    'use_qlora': True,
    'dora': True,
    'neftune': True,
    'packing': True,
}
eff = calculate_efficiency_gains(config)
print(f"⚡ {eff['vram_usage']} ({eff['vram_saved']} saved), {eff['speed_multiplier']}, {eff['quality_boost']}")

for update in train_model(config, 'sample_data.csv'):
    if update['step'] % 20 == 0:
        m = update['metrics']
        print(f"Step {m['step']}/{m['total_steps']} | Loss: {m['train_loss']} | Util: {m['data_utilization']}%")
    if update['finished']:
        print(f"✅ Done! Loss: {update['metrics']['train_loss']}")
        break


---
## 📤 Export to Ollama

- GitHub: https://github.com/arrehmanrehman28-byte/llm-forge-studio
- Issues: https://github.com/arrehmanrehman28-byte/llm-forge-studio/issues

Happy building! 🚀
